# Terra Shift — Data Collection
## 1. Environment and API Setup

In [1]:
from pathlib import Path
from dotenv import load_dotenv
import os

# Find the project folder and load its .env file
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(PROJECT_DIR / ".env")

CENSUS_API_KEY = os.getenv("CENSUS_API_KEY")

if CENSUS_API_KEY:
    print("Census API key loaded successfully.")
else:
    print("API key not found. Check that .env is in the terra-shift project folder.")

Census API key loaded successfully.


## 2. Fetching 2019 ACS Data
We are collecting:
- Total population
- Civilian employed population (age 16+)
- Total housing units
Source: US Census Bureau — ACS 5-Year Estimates, 2019.

In [2]:

import requests
import pandas as pd

# Census API endpoint for 2019 ACS 5-Year Estimates
BASE_URL = "https://api.census.gov/data/2019/acs/acs5"

# Variables we want to collect
variables = [
    "NAME",
    "B01003_001E",  # Total population
    "B01003_001M",  # Population margin of error
    "B23025_004E",  # Civilian employed population (16+)
    "B23025_004M",  # Employment margin of error
    "B25001_001E",  # Total housing units
    "B25001_001M"   # Housing units margin of error
]

params = {
    "get": ",".join(variables),
    "for": "metropolitan statistical area/micropolitan statistical area:*",
    "key": CENSUS_API_KEY
}

# Send request
response = requests.get(BASE_URL, params=params, timeout=60)

print("HTTP Status:", response.status_code)

# Check whether the request succeeded
response.raise_for_status()

# Convert the API response into a DataFrame
data = response.json()

df_2019 = pd.DataFrame(data[1:], columns=data[0])

print("Total records received:", len(df_2019))
print("\nColumns received:")
print(df_2019.columns.tolist())

print("\nFirst 5 records:")
display(df_2019.head())


HTTP Status: 200
Total records received: 938

Columns received:
['NAME', 'B01003_001E', 'B01003_001M', 'B23025_004E', 'B23025_004M', 'B25001_001E', 'B25001_001M', 'metropolitan statistical area/micropolitan statistical area']

First 5 records:


,NAME,B01003_001E,B01003_001M,B23025_004E,B23025_004M,B25001_001E,B25001_001M,metropolitan statistical area/micropolitan statistical area
0,"Big Stone Gap, VA Micro Area",42456,-555555555,14549,620,19861,225,13720
1,"Billings, MT Metro Area",179071,-555555555,90874,1095,82620,156,13740
2,"Binghamton, NY Metro Area",241874,-555555555,109310,1385,113473,358,13780
3,"Birmingham-Hoover, AL Metro Area",1085330,-555555555,498170,3186,484865,379,13820
4,"Bismarck, ND Metro Area",127503,105,70138,713,58568,189,13900


## 3. Cleaning the 2019 Metro Dataset
We will:
1. Preserve the original API response.
2. Keep only Metropolitan Statistical Areas.
3. Convert measurement columns into appropriate numeric types.
4. Handle Census special values using the same rules as our 2024 dataset.
5. Rename columns consistently.
6. Add the 2015–2019 ACS survey period.

In [3]:

# Keep an untouched copy of the original API response
df_2019_raw = df_2019.copy()

# Identify the geographic ID column
geo_column = "metropolitan statistical area/micropolitan statistical area"

# Keep only metropolitan areas
# Micropolitan areas are excluded because our model focuses on larger urban regions
df_2019_metro = df_2019[
    df_2019["NAME"].str.endswith("Metro Area", na=False)
].copy()

# Rename columns to simpler names
df_2019_metro = df_2019_metro.rename(columns={
    "NAME": "metro_name",
    geo_column: "metro_id",
    "B01003_001E": "population",
    "B01003_001M": "population_moe",
    "B23025_004E": "employed_population",
    "B23025_004M": "employment_moe",
    "B25001_001E": "housing_units",
    "B25001_001M": "housing_moe"
})

# Preserve metro ID as a string
df_2019_metro["metro_id"] = df_2019_metro["metro_id"].astype("string")

# Remove the "Metro Area" suffix from names
df_2019_metro["metro_name"] = (
    df_2019_metro["metro_name"]
    .str.replace(r"\s+Metro Area$", "", regex=True)
    .str.strip()
)

# Convert measurement columns into numeric types
numeric_columns = [
    "population",
    "population_moe",
    "employed_population",
    "employment_moe",
    "housing_units",
    "housing_moe"
]

for column in numeric_columns:
    df_2019_metro[column] = pd.to_numeric(
        df_2019_metro[column],
        errors="coerce"
    )

# Handle population MOE controlled estimates
df_2019_metro["population_moe"] = (
    df_2019_metro["population_moe"]
    .replace(-555555555, 0)
)

# Replace other Census special missing-value codes with NaN
special_codes = [
    -222222222,
    -333333333,
    -444444444,
    -555555555,
    -666666666,
    -777777777,
    -888888888,
    -999999999
]

other_moe_columns = [
    "employment_moe",
    "housing_moe"
]

for column in other_moe_columns:
    df_2019_metro[column] = (
        df_2019_metro[column]
        .replace(special_codes, float("nan"))
    )

# Convert estimate and MOE columns to nullable integer types
for column in numeric_columns:
    df_2019_metro[column] = df_2019_metro[column].astype("Int64")

# Add ACS period information
df_2019_metro["acs_period_start"] = 2015
df_2019_metro["acs_period_end"] = 2019

# Reset index after filtering
df_2019_metro = df_2019_metro.reset_index(drop=True)

# Display cleaned dataset information
print("Metropolitan areas:", len(df_2019_metro))

print("\nCleaned columns:")
print(df_2019_metro.columns.tolist())

print("\nData types:")
display(df_2019_metro.dtypes)

print("\nFirst five records:")
display(df_2019_metro.head())


Metropolitan areas: 392

Cleaned columns:
['metro_name', 'population', 'population_moe', 'employed_population', 'employment_moe', 'housing_units', 'housing_moe', 'metro_id', 'acs_period_start', 'acs_period_end']

Data types:


metro_name                str
population              Int64
population_moe          Int64
employed_population     Int64
employment_moe          Int64
housing_units           Int64
housing_moe             Int64
metro_id               string
acs_period_start        int64
acs_period_end          int64
dtype: object


First five records:


,metro_name,population,population_moe,employed_population,employment_moe,housing_units,housing_moe,metro_id,acs_period_start,acs_period_end
0,"Billings, MT",179071,0,90874,1095,82620,156,13740,2015,2019
1,"Binghamton, NY",241874,0,109310,1385,113473,358,13780,2015,2019
2,"Birmingham-Hoover, AL",1085330,0,498170,3186,484865,379,13820,2015,2019
3,"Bismarck, ND",127503,105,70138,713,58568,189,13900,2015,2019
4,"Blacksburg-Christiansburg, VA",166785,0,77138,1516,72443,344,13980,2015,2019


In [4]:

print("Dataset shape:", df_2019_metro.shape)

print("\nMissing values:")
display(df_2019_metro.isna().sum())

print("\nData types:")
display(df_2019_metro.dtypes)

print("\nBasic statistics:")
display(
    df_2019_metro[
        ["population", "employed_population", "housing_units"]
    ].describe().round(2)
)


Dataset shape: (392, 10)

Missing values:


metro_name             0
population             0
population_moe         0
employed_population    0
employment_moe         0
housing_units          0
housing_moe            0
metro_id               0
acs_period_start       0
acs_period_end         0
dtype: int64


Data types:


metro_name                str
population              Int64
population_moe          Int64
employed_population     Int64
employment_moe          Int64
housing_units           Int64
housing_moe             Int64
metro_id               string
acs_period_start        int64
acs_period_end          int64
dtype: object


Basic statistics:


,population,employed_population,housing_units
count,392.0,392.0,392.0
mean,720647.55,347540.21,297156.43
std,1612326.07,801504.43,631074.31
min,54773.0,17471.0,23974.0
25%,143253.25,63003.75,63203.75
50%,238024.5,108872.5,107094.5
75%,574636.5,268276.75,252908.25
max,19294236.0,9533728.0,7717242.0


## COMPARING WITH 2024 DATA FOR DISCREPENCY

In [6]:

from pathlib import Path
import pandas as pd

# Get the project directory
PROJECT_DIR = Path.cwd().parent

# Load the final 2024 dataset
df_2024_final = pd.read_csv(
    PROJECT_DIR / "data" / "final" / "acs_2024_metro_final.csv",
    dtype={"metro_id": "string"}
)

# Get metropolitan IDs from both datasets
metro_ids_2019 = set(df_2019_metro["metro_id"].astype("string"))
metro_ids_2024 = set(df_2024_final["metro_id"].astype("string"))

# Compare metropolitan IDs
common_metros = metro_ids_2019.intersection(metro_ids_2024)

only_2019 = metro_ids_2019 - metro_ids_2024
only_2024 = metro_ids_2024 - metro_ids_2019

# Display comparison results
print("2019 metropolitan areas:", len(metro_ids_2019))
print("2024 metropolitan areas:", len(metro_ids_2024))
print("Common metropolitan areas:", len(common_metros))

print("\nMetros only in 2019:")
display(
    df_2019_metro[
        df_2019_metro["metro_id"].isin(only_2019)
    ][["metro_id", "metro_name"]]
)

print("\nMetros only in 2024:")
display(
    df_2024_final[
        df_2024_final["metro_id"].isin(only_2024)
    ][["metro_id", "metro_name"]]
)


2019 metropolitan areas: 392
2024 metropolitan areas: 392
Common metropolitan areas: 376

Metros only in 2019:


,metro_id,metro_name
7,14100,"Bloomsburg-Berwick, PA"
16,15380,"Buffalo-Cheektowaga, NY"
19,15680,"California-Lexington Park, MD"
25,16060,"Carbondale-Marion, IL"
40,17460,"Cleveland-Elyria, OH"
56,19060,"Cumberland, MD-WV"
59,19180,"Danville, IL"
70,20700,"East Stroudsburg, PA"
183,36140,"Ocean City, NJ"
197,38220,"Pine Bluff, AR"



Metros only in 2024:


,metro_id,metro_name
12,11200,"Amherst Town-Northampton, MA"
47,14580,"Bozeman, MT"
72,17410,"Cleveland, OH"
101,20580,"Eagle Pass, TX"
152,25740,"Helena, MT"
182,28450,"Kenosha, WI"
186,28880,"Kiryas Joel-Poughkeepsie-Newburgh, NY"
206,30500,"Lexington Park, MD"
234,33500,"Minot, ND"
266,37140,"Paducah, KY-IL"
